# Notebook 03 - LSTM Du Doan Nhip Tim

Model nay hoc chuoi `heart_rate` tu `health_data_augmented.csv` va du doan gia tri HR tiep theo tu cua so 24 mau.

In [ ]:
# Cell 1: Chuan bi sequences
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('health_data_augmented.csv'),
    Path('../colab_export/health_data_augmented.csv'),
    Path('ai_training/colab_export/health_data_augmented.csv'),
    Path('/content/health_data_augmented.csv'),
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Khong tim thay health_data_augmented.csv. Hay upload file vao Colab hoac chay export_colab_notebook_data.py truoc.')

df = pd.read_csv(data_path)
df = df.dropna(subset=['heart_rate']).reset_index(drop=True)
hr = df['heart_rate'].values.reshape(-1, 1).astype(np.float32)

WINDOW = 24

def make_sequences(data, w):
    X, y = [], []
    for i in range(len(data) - w):
        X.append(data[i:i + w])
        y.append(data[i + w])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = make_sequences(hr, WINDOW)
split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print('Data:', data_path)
print('Rows:', len(df))
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:', X_val.shape, 'y_val:', y_val.shape)

In [ ]:
# Cell 2: Build model LSTM
model = keras.Sequential([
    keras.layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, 1)),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# Cell 3: Train
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('LSTM loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 4: Danh gia va luu
val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
print(f'Val MAE: {val_mae:.4f} bpm')
print(f'Val MAE quy doi theo cong thuc notebook goc: ~{val_mae * 220:.1f} bpm sai so')

model.save('lstm_heart_predict.h5')
print('Saved: lstm_heart_predict.h5')

In [ ]:
# Cell 5: Ve thu du doan
pred = model.predict(X_val[:100], verbose=0).reshape(-1)
truth = y_val[:100].reshape(-1)

plt.figure(figsize=(12, 4))
plt.plot(truth, label='Thuc te')
plt.plot(pred, label='Du doan')
plt.title('LSTM HR prediction')
plt.ylabel('bpm')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 6: Download tu Colab ve may va copy vao models/ cua project neu co
artifact_candidates = {
    'isolation_forest.pkl': [Path('isolation_forest.pkl'), Path('../colab_export/isolation_forest.pkl'), Path('ai_training/colab_export/isolation_forest.pkl')],
    'scaler.pkl': [Path('scaler.pkl'), Path('../colab_export/scaler.pkl'), Path('ai_training/colab_export/scaler.pkl')],
    'lstm_heart_predict.h5': [Path('lstm_heart_predict.h5')],
}

for target_name, candidates in artifact_candidates.items():
    source = next((p for p in candidates if p.exists()), None)
    if source is not None and source.name != target_name:
        shutil.copy2(source, target_name)

try:
    from google.colab import files
    files.download('isolation_forest.pkl')
    files.download('lstm_heart_predict.h5')
    files.download('scaler.pkl')
except Exception as exc:
    print('Khong phai Colab hoac khong download duoc:', exc)

candidate_model_dirs = [
    Path('/content/aiot_health/models'),
    Path('/content/drive/MyDrive/aiot_health/models'),
    Path('../../models'),
    Path('models'),
]

for model_dir in candidate_model_dirs:
    if model_dir.exists():
        for name in ['isolation_forest.pkl', 'lstm_heart_predict.h5', 'scaler.pkl']:
            if Path(name).exists():
                shutil.copy2(name, model_dir / name)
        print(f'Copied artifacts to: {model_dir.resolve()}')
        break
else:
    print('Chua tim thay thu muc models/. Hay copy cac file vua download vao models/ cua project.')